In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 主样本表：后面 Logit 建模主要用这张
sample_7d = pd.read_csv(
    'step2_click_sample_labeled_7d.csv',
    parse_dates=['click_date', 'first_click_ts', 'last_click_ts', 'order_time'],
    low_memory=False
)

# 描述统计结果表：后面解释结果时会用到
channel_summary = pd.read_csv('step3_channel_summary.csv', low_memory=False)
click_bucket_summary = pd.read_csv('step3_click_bucket_summary.csv', low_memory=False)
focus_summary = pd.read_csv('step3_focus_summary.csv', low_memory=False)
channel_type_summary = pd.read_csv('step3_channel_type_summary.csv', low_memory=False)

channel_clickcnt_summary = pd.read_csv('step3_channel_clickcnt_summary.csv', low_memory=False)
channel_clickcnt_pivot = pd.read_csv('step3_channel_clickcnt_pivot.csv', low_memory=False)

channel_focus_summary = pd.read_csv('step3_channel_focus_summary.csv', low_memory=False)
channel_focus_pivot = pd.read_csv('step3_channel_focus_pivot.csv', low_memory=False)

print('sample_7d:', sample_7d.shape)
print('channel_summary:', channel_summary.shape)
print('click_bucket_summary:', click_bucket_summary.shape)
print('focus_summary:', focus_summary.shape)
print('channel_type_summary:', channel_type_summary.shape)
print('channel_clickcnt_summary:', channel_clickcnt_summary.shape)
print('channel_clickcnt_pivot:', channel_clickcnt_pivot.shape)
print('channel_focus_summary:', channel_focus_summary.shape)
print('channel_focus_pivot:', channel_focus_pivot.shape)

sample_7d: (213050, 41)
channel_summary: (5, 9)
click_bucket_summary: (6, 3)
focus_summary: (2, 5)
channel_type_summary: (10, 6)
channel_clickcnt_summary: (30, 7)
channel_clickcnt_pivot: (6, 6)
channel_focus_summary: (10, 7)
channel_focus_pivot: (5, 4)


In [3]:
model_df = sample_7d.copy()

# 只保留第一版建模最核心的变量
model_df = model_df[
    [
        'label_7d',
        'channel',
        'type',
        'sku_focus_flag',
        'repeat_same_sku_flag',
        'click_cnt',
        'user_day_distinct_skus',
        'user_day_distinct_channels',
        'is_plus',
        'purchase_power',
        'city_level',
        'gender',
        'age'
    ]
].copy()

# 删除关键变量缺失值
model_df = model_df.dropna().copy()

print(model_df.shape)
model_df.head()

(213050, 13)


,label_7d,channel,type,sku_focus_flag,repeat_same_sku_flag,click_cnt,user_day_distinct_skus,user_day_distinct_channels,is_plus,purchase_power,city_level,gender,age
0,1,app,1,0,0,1,2,1,1,2,1,F,26-35
1,0,app,2,1,0,1,1,1,0,2,4,F,26-35
2,0,app,1,1,0,1,1,1,1,2,4,F,26-35
3,0,app,2,1,0,1,1,1,0,3,1,M,16-25
4,1,app,1,0,0,1,2,1,0,4,4,M,26-35


In [4]:
model1 = smf.logit(
    'label_7d ~ C(channel)',
    data=model_df
).fit()

print(model1.summary())

Optimization terminated successfully.
         Current function value: 0.402425
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               label_7d   No. Observations:               213050
Model:                          Logit   Df Residuals:                   213045
Method:                           MLE   Df Model:                            4
Date:                Mon, 13 Apr 2026   Pseudo R-squ.:                0.003054
Time:                        11:47:53   Log-Likelihood:                -85737.
converged:                       True   LL-Null:                       -85999.
Covariance Type:            nonrobust   LLR p-value:                2.233e-112
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               -1.8582      0.007   -268.024      0.000      -1.872      -1.845

In [10]:
model2_fix = smf.logit(
    "label_7d ~ C(channel, Treatment(reference='app')) + C(type) + is_plus + C(purchase_power) + C(city_level) + C(gender) + C(age)",
    data=model_df
).fit(method='lbfgs', maxiter=200)

print(model2_fix.summary())

                           Logit Regression Results                           
Dep. Variable:               label_7d   No. Observations:               213050
Model:                          Logit   Df Residuals:                   213025
Method:                           MLE   Df Model:                           24
Date:                Mon, 13 Apr 2026   Pseudo R-squ.:                 0.01397
Time:                        12:11:36   Log-Likelihood:                -84798.
converged:                       True   LL-Null:                       -85999.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------
Intercept                                           -1.7462      0.037    -47.830      0.000      -1.818      -1.675
C(channel, Treatm

In [11]:
base_formula = (
    "label_7d ~ "
    "C(channel, Treatment(reference='app')) + "
    "C(type) + is_plus + "
    "C(purchase_power) + C(city_level) + "
    "C(gender) + C(age)"
)

m3_a = smf.logit(
    base_formula + " + sku_focus_flag",
    data=model_df
).fit(method='lbfgs', maxiter=300)

m3_b = smf.logit(
    base_formula + " + repeat_same_sku_flag",
    data=model_df
).fit(method='lbfgs', maxiter=300)

m3_c = smf.logit(
    base_formula + " + click_cnt",
    data=model_df
).fit(method='lbfgs', maxiter=300)

m3_d = smf.logit(
    base_formula + " + user_day_distinct_skus",
    data=model_df
).fit(method='lbfgs', maxiter=300)

m3_e = smf.logit(
    base_formula + " + user_day_distinct_channels",
    data=model_df
).fit(method='lbfgs', maxiter=300)

print('m3_a converged =', m3_a.mle_retvals['converged'])
print('m3_b converged =', m3_b.mle_retvals['converged'])
print('m3_c converged =', m3_c.mle_retvals['converged'])
print('m3_d converged =', m3_d.mle_retvals['converged'])
print('m3_e converged =', m3_e.mle_retvals['converged'])

m3_a converged = True
m3_b converged = True
m3_c converged = True
m3_d converged = True
m3_e converged = True


In [12]:
model3_focus = smf.logit(
    base_formula + " + sku_focus_flag + user_day_distinct_skus",
    data=model_df
).fit(method='lbfgs', maxiter=300)

print(model3_focus.summary())

                           Logit Regression Results                           
Dep. Variable:               label_7d   No. Observations:               213050
Model:                          Logit   Df Residuals:                   213023
Method:                           MLE   Df Model:                           26
Date:                Mon, 13 Apr 2026   Pseudo R-squ.:                  0.1051
Time:                        12:20:16   Log-Likelihood:                -76964.
converged:                       True   LL-Null:                       -85999.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------
Intercept                                           -1.6862      0.041    -41.455      0.000      -1.766      -1.607
C(channel, Treatm

In [13]:
model3_repeat = smf.logit(
    base_formula + " + repeat_same_sku_flag + click_cnt",
    data=model_df
).fit(method='lbfgs', maxiter=300)

print(model3_repeat.summary())

                           Logit Regression Results                           
Dep. Variable:               label_7d   No. Observations:               213050
Model:                          Logit   Df Residuals:                   213023
Method:                           MLE   Df Model:                           26
Date:                Mon, 13 Apr 2026   Pseudo R-squ.:                 0.03790
Time:                        12:19:15   Log-Likelihood:                -82740.
converged:                       True   LL-Null:                       -85999.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------
Intercept                                           -2.1391      0.038    -56.826      0.000      -2.213      -2.065
C(channel, Treatm

In [14]:
model3_channelmix = smf.logit(
    base_formula + " + user_day_distinct_channels",
    data=model_df
).fit(method='lbfgs', maxiter=300)

print(model3_channelmix.summary())

                           Logit Regression Results                           
Dep. Variable:               label_7d   No. Observations:               213050
Model:                          Logit   Df Residuals:                   213024
Method:                           MLE   Df Model:                           25
Date:                Mon, 13 Apr 2026   Pseudo R-squ.:                 0.01407
Time:                        12:20:38   Log-Likelihood:                -84789.
converged:                       True   LL-Null:                       -85999.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------
Intercept                                           -1.6591      0.043    -38.739      0.000      -1.743      -1.575
C(channel, Treatm

In [15]:
models = {
    'model2_base': model2,
    'model3_focus': model3_focus,
    'model3_repeat': model3_repeat,
    'model3_channelmix': model3_channelmix
}

key_vars = [
    "C(channel, Treatment(reference='app'))[T.mobile]",
    "C(channel, Treatment(reference='app'))[T.others]",
    "C(channel, Treatment(reference='app'))[T.pc]",
    "C(channel, Treatment(reference='app'))[T.wechat]",
    "C(type)[T.2]",
    "is_plus",
    "sku_focus_flag",
    "user_day_distinct_skus",
    "repeat_same_sku_flag",
    "click_cnt",
    "user_day_distinct_channels"
]

coef_table = pd.DataFrame({
    name: model.params.reindex(key_vars)
    for name, model in models.items()
})

pval_table = pd.DataFrame({
    name: model.pvalues.reindex(key_vars)
    for name, model in models.items()
})

print('系数表：')
display(coef_table)

print('P值表：')
display(pval_table)

系数表：


,model2_base,model3_focus,model3_repeat,model3_channelmix
"C(channel, Treatment(reference='app'))[T.mobile]",0.624769,0.595735,0.739334,0.646547
"C(channel, Treatment(reference='app'))[T.others]",0.363035,0.558179,0.457951,0.448840
"C(channel, Treatment(reference='app'))[T.pc]",0.358963,0.534599,0.466590,0.410753
"C(channel, Treatment(reference='app'))[T.wechat]",0.044121,0.085749,0.099148,0.048494
C(type)[T.2],-0.547216,-0.431315,-0.472362,-0.546312
is_plus,-0.086010,-0.059153,-0.088677,-0.082254
sku_focus_flag,NaN,1.145007,NaN,NaN
user_day_distinct_skus,NaN,-0.063678,NaN,NaN
repeat_same_sku_flag,NaN,NaN,0.022024,NaN
click_cnt,NaN,NaN,0.118924,NaN


P值表：


,model2_base,model3_focus,model3_repeat,model3_channelmix
"C(channel, Treatment(reference='app'))[T.mobile]",1.508267e-55,6.128983e-46,2.963266e-76,2.360350e-58
"C(channel, Treatment(reference='app'))[T.others]",3.675839e-11,4.352498e-22,9.946863e-17,1.617213e-14
"C(channel, Treatment(reference='app'))[T.pc]",3.634312e-44,9.895650e-87,9.393639e-71,1.666612e-46
"C(channel, Treatment(reference='app'))[T.wechat]",1.017004e-01,2.240578e-03,2.750264e-04,7.249115e-02
C(type)[T.2],0.000000e+00,9.392839e-187,1.397566e-234,0.000000e+00
is_plus,3.148799e-08,2.637917e-04,1.977634e-08,1.295398e-07
sku_focus_flag,NaN,0.000000e+00,NaN,NaN
user_day_distinct_skus,NaN,0.000000e+00,NaN,NaN
repeat_same_sku_flag,NaN,NaN,3.427449e-01,NaN
click_cnt,NaN,NaN,0.000000e+00,NaN


In [17]:
model4_interact = smf.logit(
    "label_7d ~ "
    "C(channel, Treatment(reference='app')) * C(type) + "
    "is_plus + C(purchase_power) + C(city_level) + C(gender) + C(age)",
    data=model_df
).fit(method='lbfgs', maxiter=300)

print(model4_interact.summary())

                           Logit Regression Results                           
Dep. Variable:               label_7d   No. Observations:               213050
Model:                          Logit   Df Residuals:                   213021
Method:                           MLE   Df Model:                           28
Date:                Mon, 13 Apr 2026   Pseudo R-squ.:                 0.01440
Time:                        13:37:01   Log-Likelihood:                -84760.
converged:                       True   LL-Null:                       -85999.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------------
Intercept                                                        -1.7340      0.037    -47.335      0.000   

In [18]:
model4_interact_focus = smf.logit(
    "label_7d ~ "
    "C(channel, Treatment(reference='app')) * C(type) + "
    "is_plus + C(purchase_power) + C(city_level) + C(gender) + C(age) + "
    "sku_focus_flag + user_day_distinct_skus",
    data=model_df
).fit(method='lbfgs', maxiter=300)

print(model4_interact_focus.summary())

                           Logit Regression Results                           
Dep. Variable:               label_7d   No. Observations:               213050
Model:                          Logit   Df Residuals:                   213019
Method:                           MLE   Df Model:                           30
Date:                Mon, 13 Apr 2026   Pseudo R-squ.:                  0.1053
Time:                        14:14:06   Log-Likelihood:                -76940.
converged:                       True   LL-Null:                       -85999.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------------
Intercept                                                        -1.6738      0.041    -41.033      0.000   